# Inversion of average LE GCL-down and RCS GCL-down traces with forward model validation

## Shared functions

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import matplotlib.pyplot as plt
from collections import OrderedDict

from data_processing.common_utils import CommonUtils
from data_processing.analysis_stage.convert_to_1d import Res2DInvOutputProcessor

# shared style settings
AXIS_LABEL_FONT_SIZE = 18
TICK_LABEL_FONT_SIZE = 16
LEGEND_FONT_SIZE = 18
LAYER_LABEL_FONT_SIZE = 14
LINE_WIDTH_SCALE = 1.3
LEGEND_HANDLE_WIDTH_SCALE = 1
LEGEND_HANDLE_LENGTH = 2
SHARED_X_RANGE = [0, 5000]

def _bump_style(ax, legend_kwargs=None):
    ax.xaxis.label.set_fontsize(AXIS_LABEL_FONT_SIZE)
    ax.yaxis.label.set_fontsize(AXIS_LABEL_FONT_SIZE)
    ax.tick_params(axis="both", labelsize=TICK_LABEL_FONT_SIZE)
    for line in ax.get_lines():
       line.set_linewidth(line.get_linewidth() * LINE_WIDTH_SCALE)
    for container in ax.containers:
       if not hasattr(container, "lines"):
          continue
       data_line, cap_lines, bar_line_cols = container.lines
       if data_line is not None:
          data_line.set_linewidth(data_line.get_linewidth() * LINE_WIDTH_SCALE)
       for cap in cap_lines:
          cap.set_linewidth(cap.get_linewidth() * LINE_WIDTH_SCALE)
       for coll in bar_line_cols:
          coll.set_linewidth(coll.get_linewidth()[0] * LINE_WIDTH_SCALE)
    legend = ax.get_legend()
    if legend is not None:
       handles, labels = ax.get_legend_handles_labels()
       legend.remove()
       legend_options = {'handlelength': LEGEND_HANDLE_LENGTH, 'fontsize': LEGEND_FONT_SIZE}
       legend_options.update(legend_kwargs or {})
       legend = ax.legend(handles, labels, **legend_options)
       for handle in getattr(legend, "legend_handles", None) or getattr(legend, "legendHandles", []):
           lw = handle.get_linewidth()
           if hasattr(lw, "__len__"):
              handle.set_linewidth([w * LEGEND_HANDLE_WIDTH_SCALE for w in lw])
           else:
              handle.set_linewidth(lw * LEGEND_HANDLE_WIDTH_SCALE)
           if hasattr(handle, "set_markersize"):
              handle.set_markersize(6)
           if hasattr(handle, "set_markeredgewidth"):
              handle.set_markeredgewidth(1)


def _run_le(ax, x_range_to_plot=None, layer_label_font_size=None, orientation='horizontal'):
	layer_label_font_size = layer_label_font_size if layer_label_font_size is not None else LAYER_LABEL_FONT_SIZE
	root_folder = (r".../aggregated_long_evans_data")

	layers = {"0": "", "-19": "", "-67": "", "-146": "", "-200": ""}
	labels = ["NFL+GCL", "IPL", "INL+OPL+ONL", "IS+OS", "PBS"]
	left_side_labels = ["NFL+GCL", "INL+OPL+ONL"]
	depth_range = [-220, 0]
	x_range_to_use = [300, 500]

	header_files = CommonUtils.find_header_files(
		root_folder,
		keyword='inversion_result',
		search_depth=1,
		sort_by_index=True
	)

	keyword_combinations = [
		["sim_sample", "Res2DInv_decoded", "4layers_no_ap", "0fixed_bound_damp_2depth"],
		["sim", "average_trace_all_first", "600um_no_ap", "0fixed_bound_damp_2depth"],
	]
	filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

	ground_truth_list = []
	ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))
	ground_truth_4l = OrderedDict(sorted({"0": "2687", "-19": "1628", "-67": "2254", "-146": "1026", "-200": "60"}.items
										 (), key=lambda x: int(x[0]), reverse=True))
	ground_truth_list.append(ground_truth_4l)
	ground_truth_list.append(ground_truth_experimental)

	processor = Res2DInvOutputProcessor()
	output_data = processor.run_1d_conversion(
		filtered_files,
		indices_to_process=[2000],
		x_range=x_range_to_use,
		depth_range=depth_range,
		labels=labels,
		left_side_labels=left_side_labels,
		layers=layers,
		ground_truth=ground_truth_list,
		calculate_average_resistivity=True,
		plot=True,
		plot_all_together=True,
		layer_bound_ground_truth=True,
		x_range_to_plot=x_range_to_plot if x_range_to_plot is not None else [0, 4700],
		colors=["#1B9E77", "#2E5EAA"],
		legend_labels=['Forward Model', 'Experimental'],
		kohm_labels=True,
		ax=ax,
		legend_position='best',
		layer_label_font_size=layer_label_font_size,
		orientation=orientation,
	)
	return {
		'depth_range': depth_range,
		'output_data': output_data,
		'layers': layers,
		'labels': labels,
		'left_side_labels': left_side_labels,
		'ground_truth_list': ground_truth_list,
		'processor': processor,
	}


def _run_rcs(ax, x_range_to_plot=None, layer_label_font_size=None, orientation='horizontal'):
	layer_label_font_size = layer_label_font_size if layer_label_font_size is not None else LAYER_LABEL_FONT_SIZE
	root_folder = (r".../aggregated_rcs_data")

	layers = {"0": "", "-24.5": "", "-91.5": "", "-126": ""}
	labels = ["NFL+GCL", "IPL", "INL", "PBS"]
	left_side_labels = ["NFL+GCL"]
	depth_range = [-220, 0]
	x_range_to_use = [300, 500]

	header_files = CommonUtils.find_header_files(
		root_folder,
		keyword='inversion_result',
		search_depth=1,
		sort_by_index=True
	)

	keyword_combinations = [
		["sim", "Res2DInv_decoded_RCS", "3layers_no_ap", "6samples"],
		["sim", "ML_decoded_RCS", "3layers", "no_ap", "0fixed_bound_damp_2depth", "6samples"],
		["sim", "average_trace_all_first", "600um_no_ap", "0fixed_bound_damp_2depth_6samples"],
	]
	filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

	ground_truth_list = []
	ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: float(x[0]), reverse=True))
	ground_truth_3l = OrderedDict(sorted({"0": "4236", "-24.38": "1325", "-91.88": "1517", "-126": "60"}.items(),
	 									 key=lambda x: float(x[0]), reverse=True))
	ground_truth_3l_modified = OrderedDict(sorted({"0": "3373", "-24.38": "1120", "-91.88": "4045", "-126": "60"}
												  .items(),
										 key=lambda x: float(x[0]), reverse=True))
	ground_truth_list.append(ground_truth_3l)
	ground_truth_list.append(ground_truth_3l_modified)
	ground_truth_list.append(ground_truth_experimental)

	processor = Res2DInvOutputProcessor()
	output_data = processor.run_1d_conversion(
		filtered_files,
		indices_to_process=[4000],
		x_range=x_range_to_use,
		depth_range=depth_range,
		labels=labels,
		left_side_labels=left_side_labels,
		layers=layers,
		ground_truth=ground_truth_list,
		calculate_average_resistivity=True,
		plot=True,
		plot_all_together=True,
		layer_bound_ground_truth=True,
		x_range_to_plot=x_range_to_plot if x_range_to_plot is not None else [0, 4700],
		colors=["#D98296", "#3EA136", "#C23B5A"],
		legend_labels=['Inv Forward Model', 'ML Forward Model', 'Experimental'],
		kohm_labels=True,
		legend_position='best',
		ax=ax,
		layer_label_font_size=layer_label_font_size,
		orientation=orientation,
	)
	return {
		'depth_range': depth_range,
		'output_data': output_data,
		'layers': layers,
		'labels': labels,
		'left_side_labels': left_side_labels,
		'ground_truth_list': ground_truth_list,
		'processor': processor,
	}


# --- Layer extraction analysis (default extraction method: 'extremum' for every layer) ---

def _run_layer_extraction_analysis(run_fn, name, depth_range_to_use=None, layer_methods=None):
	"""Run the shared per-layer resistivity extraction (the analysis we used to
	run against each validation dataset), reusing the exact same file list and
	ground truth that `run_fn` (either `_run_le` or `_run_rcs`) uses for its
	plot. Any layer not listed in `layer_methods` defaults to 'extremum'.
	Prints, and returns, the per-layer average error against ground truth."""
	fig, throwaway_ax = plt.subplots()
	run_result = run_fn(throwaway_ax)
	plt.close(fig)

	depth_range = depth_range_to_use if depth_range_to_use is not None else run_result['depth_range']
	all_results = run_result['processor'].extract_resistivity_values_from_set_of_1d_traces(
		run_result['output_data'], run_result['layers'], run_result['labels'], depth_range,
		plot=True, layer_methods=layer_methods, left_side_labels=run_result.get('left_side_labels'),
	)
	per_layer_avg = run_result['processor'].average_error_by_layer(
		all_results, run_result['ground_truth_list'], run_result['labels']
	)
	print(f"{name} layer extraction (default method: extremum):")
	for label, avg_error in per_layer_avg.items():
		print(f"  {label}: {avg_error:.1f}%")
	return all_results



## LE GCL-down

In [ ]:
X_RANGE = [0, 3000]
le_fig, le_ax_single = plt.subplots(figsize=(12, 8))
_run_le(le_ax_single, x_range_to_plot=X_RANGE)
_bump_style(le_ax_single, legend_kwargs={'loc': 'lower right', 'bbox_to_anchor': (0.78, 0.02), 'fontsize': 16})
plt.show()


le_layer_extraction_results = _run_layer_extraction_analysis(_run_le, "LE")

## RCS GCL-down

In [ ]:
X_RANGE = [0, 5700]
rcs_fig, rcs_ax_single = plt.subplots(figsize=(12, 8))
_run_rcs(rcs_ax_single, x_range_to_plot=X_RANGE)
_bump_style(rcs_ax_single, legend_kwargs={'loc': 'lower right', 'bbox_to_anchor': (0.78, 0.02), 'fontsize': 16})
plt.show()

rcs_layer_extraction_results = _run_layer_extraction_analysis(_run_rcs, "RCS")


# Decoding Accuracy

## LE GCL-down

### All COMSOL Test Dataset

In [ ]:


from data_processing.common_utils import CommonUtils
from data_processing.analysis_stage.convert_to_1d import Res2DInvOutputProcessor


def _extract_file_name(header_file_entry):
    """Best-effort extraction of a file name from a header_files entry."""
    if isinstance(header_file_entry, dict):
        for key in ("file_path", "header_file", "path", "file"):
            if key in header_file_entry:
                return os.path.basename(str(header_file_entry[key]))
    return os.path.basename(str(header_file_entry))


def build_ground_truth_from_file_name(file_name, labels, depth_key_by_label, token_overrides=None):
    """Build a ground truth dict from each layer's absolute resistivity value
    encoded directly in the file name, e.g.
    "..._NFL_GCL_62.4_IPL_5128_nu_2761.5_PR_1756_saline_60_..." gives
    NFL+GCL=62.4, IPL=5128, INL+OPL+ONL=2761.5, IS+OS=1756, PBS=60.

    token_overrides maps a label to the literal token used in file names, for
    layers whose file-name token doesn't match label.replace("+", "_").
    Defaults to the known short/alternate tokens used in these files:
    INL+OPL+ONL -> "nu", IS+OS -> "PR", PBS -> "saline".
    """
    if token_overrides is None:
        token_overrides = {
            "INL+OPL+ONL": "nu",
            "IS+OS": "PR",
            "PBS": "saline",
        }
    ground_truth = OrderedDict()
    for label in labels:
        key = depth_key_by_label[label]
        token = token_overrides.get(label, label.replace("+", "_"))
        match = re.search(rf"{re.escape(token)}_(\d+(?:\.\d+)?)", file_name, re.IGNORECASE)
        if not match:
            raise ValueError(f"could not find token '{token}' (for label '{label}') in file name: {file_name}")
        ground_truth[key] = match.group(1)
    return ground_truth


def build_legend_label_from_file_name(file_name, labels, token_overrides=None):
    """Pull a compact 'token:value' summary for each known layer token out of
    the file name, e.g. "NFL_GCL:62.4 IPL:5128 nu:2761.5 PR:1756 saline:60",
    or the raw file name if none of the known tokens are found."""
    if token_overrides is None:
        token_overrides = {
            "INL+OPL+ONL": "nu",
            "IS+OS": "PR",
            "PBS": "saline",
        }
    parts = []
    for label in labels:
        token = token_overrides.get(label, label.replace("+", "_"))
        match = re.search(rf"{re.escape(token)}_(\d+(?:\.\d+)?)", file_name, re.IGNORECASE)
        if match:
            parts.append(f"{token}:{match.group(1)}")
    return " ".join(parts) if parts else file_name


root_folder = (r".../comsol_8000_test_dataset_exported_results")

# LE settings
layers = {"0": "", "-19": "", "-67": "", "-146": "", "-200": ""}
labels = ["NFL+GCL", "IPL", "INL+OPL+ONL", "IS+OS", "PBS"]
left_side_labels = ["NFL+GCL"]
depth_range = [-200, 0]
x_range_to_use = [300, 500]

# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='comsol_testset',
    search_depth=0,
    sort_by_index=False
)

# filter by keyword combinations - no saline fixation
keyword_combinations = [
    ["comsol", "8000"],
]

# depth keys are sorted the same way as `labels` (deepest depth last), so they line up 1:1
depth_key_by_label = dict(zip(labels, sorted(layers.keys(), key=lambda k: int(k), reverse=True)))

# create one ground truth entry per filtered file, read directly from the file name
ground_truth_list = [
    build_ground_truth_from_file_name(_extract_file_name(f), labels, depth_key_by_label)
    for f in header_files
]

# create one legend label per filtered file, based on the file name
legend_labels_list = [
    build_legend_label_from_file_name(_extract_file_name(f), labels)
    for f in header_files
]


# convert data to 1D traces
processor = Res2DInvOutputProcessor()
output_data = processor.run_1d_conversion(
    header_files,
    indices_to_process=[],
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    left_side_labels=left_side_labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=False,
    plot=False,
    plot_all_together=False,
    layer_bound_ground_truth=True,
    x_range_to_plot=[0, 3500],
    # colors = ["#709EE4", "#C37F18", "#5BA23A"],
    legend_labels=legend_labels_list,
    kohm_labels=True,
#   colors= ["black", "darkgreen"]
)

# extract values
layer_methods = {
    "NFL+GCL": "extremum",
    "IPL": "extremum",
    "INL+OPL+ONL": "extremum",
    "IS+OS": "extremum",
    "PBS": "layer_average",
}
all_results = processor.extract_resistivity_values_from_set_of_1d_traces(output_data, layers, labels, [-215, 0], plot=True,
																		 layer_methods=layer_methods, max_plots=30)

per_layer_avg = processor.average_error_by_layer(all_results, ground_truth_list, labels)
print(f"Number of processed traces: {len(all_results)}")
for label, avg_error in per_layer_avg.items():
    print(f"{label}: {avg_error:.1f}%")

## RCS GCL-down

### All COMSOL Test Dataset

In [ ]:
import os
import re
from collections import OrderedDict

from data_processing.common_utils import CommonUtils
from data_processing.analysis_stage.convert_to_1d import Res2DInvOutputProcessor


def _extract_file_name(header_file_entry):
    """Best-effort extraction of a file name from a header_files entry."""
    if isinstance(header_file_entry, dict):
        for key in ("file_path", "header_file", "path", "file"):
            if key in header_file_entry:
                return os.path.basename(str(header_file_entry[key]))
    return os.path.basename(str(header_file_entry))


def build_ground_truth_from_file_name(file_name, labels, depth_key_by_label, token_overrides=None):
    """Build a ground truth dict from each layer's absolute resistivity value
    encoded directly in the file name, e.g.
    "..._NFL_GCL_62.4_IPL_5128_nu_2761.5_PR_1756_saline_60_..." gives
    NFL+GCL=62.4, IPL=5128, INL+OPL+ONL=2761.5, IS+OS=1756, PBS=60.

    token_overrides maps a label to the literal token used in file names, for
    layers whose file-name token doesn't match label.replace("+", "_").
    Defaults to the known short/alternate tokens used in these files:
    INL+OPL+ONL -> "nu", IS+OS -> "PR", PBS -> "saline".
    """
    if token_overrides is None:
        token_overrides = {
			"INL": "nu",
            "INL+OPL+ONL": "nu",
            "IS+OS": "PR",
            "PBS": "saline",
        }
    ground_truth = OrderedDict()
    for label in labels:
        key = depth_key_by_label[label]
        token = token_overrides.get(label, label.replace("+", "_"))
        match = re.search(rf"{re.escape(token)}_(\d+(?:\.\d+)?)", file_name, re.IGNORECASE)
        if not match:
            raise ValueError(f"could not find token '{token}' (for label '{label}') in file name: {file_name}")
        ground_truth[key] = match.group(1)
    return ground_truth


def build_legend_label_from_file_name(file_name, labels, token_overrides=None):
    """Pull a compact 'token:value' summary for each known layer token out of
    the file name, e.g. "NFL_GCL:62.4 IPL:5128 nu:2761.5 PR:1756 saline:60",
    or the raw file name if none of the known tokens are found."""
    if token_overrides is None:
        token_overrides = {
            "INL+OPL+ONL": "nu",
            "IS+OS": "PR",
            "PBS": "saline",
        }
    parts = []
    for label in labels:
        token = token_overrides.get(label, label.replace("+", "_"))
        match = re.search(rf"{re.escape(token)}_(\d+(?:\.\d+)?)", file_name, re.IGNORECASE)
        if match:
            parts.append(f"{token}:{match.group(1)}")
    return " ".join(parts) if parts else file_name


root_folder = (r".../comsol_8000_test_dataset_exported_results")

# LE settings
layers = {"0": "", "-24.5": "", "-91.5": "", "-126": ""}
labels = ["NFL+GCL", "IPL", "INL", "PBS"]
left_side_labels = ["NFL+GCL"]
depth_range = [-200, 0]
x_range_to_use = [300, 500]

# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='comsol_testset',
    search_depth=0,
    sort_by_index=False
)

# filter by keyword combinations - no saline fixation
keyword_combinations = [
    ["comsol", "8000"],
]

# depth keys are sorted the same way as `labels` (deepest depth last), so they line up 1:1
depth_key_by_label = dict(zip(labels, sorted(layers.keys(), key=lambda k: float(k), reverse=True)))

# create one ground truth entry per filtered file, read directly from the file name
ground_truth_list = [
    build_ground_truth_from_file_name(_extract_file_name(f), labels, depth_key_by_label)
    for f in header_files
]

# create one legend label per filtered file, based on the file name
legend_labels_list = [
    build_legend_label_from_file_name(_extract_file_name(f), labels)
    for f in header_files
]

# convert data to 1D traces
processor = Res2DInvOutputProcessor()
output_data = processor.run_1d_conversion(
    header_files,
    indices_to_process=[],
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    left_side_labels=left_side_labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=False,
    plot=False,
    plot_all_together=False,
    layer_bound_ground_truth=True,
    x_range_to_plot=[0, 3500],
    # colors = ["#709EE4", "#C37F18", "#5BA23A"],
    legend_labels=legend_labels_list,
    kohm_labels=True,
#   colors= ["black", "darkgreen"]
)

layer_methods = {
    "NFL+GCL": "extremum",
    "IPL": "extremum",
    "INL": "extremum",
    "PBS": "layer_average",
}
all_results = processor.extract_resistivity_values_from_set_of_1d_traces(output_data, layers, labels, [-215, 0], plot=True,
																		 layer_methods=layer_methods, max_plots=30)

per_layer_avg = processor.average_error_by_layer(all_results, ground_truth_list, labels)
print(f"Number of processed traces: {len(all_results)}")
for label, avg_error in per_layer_avg.items():
    print(f"{label}: {avg_error:.1f}%")

# Run conversion on individual experiments

## Common Configuration

In [ ]:
from collections import OrderedDict

# define root folder
root_folder = (r".../processed_long_evans_data")

# set x-range
x_range_to_use = [300, 500]

# assign settings based on species
if "rcs" in root_folder:
	# RCS settings
    layers = {"0": "", "-24": "", "-92": "", "-126": ""}
    labels = ["NFL+GCL", "IPL", "INL+OPL+ONL", "PBS"]
    depth_range = [-126, 0]

elif "saline" in root_folder:
	# saline settings
    layers = {"0": "745"}
    labels = ["SALINE"]
    depth_range = [-240, 0]

else:
	# LE settings
    layers = {"0": "", "-19": "", "-67": "", "-146": "", "-200": ""}
    labels = ["NFL+GCL", "IPL", "INL+OPL+ONL", "IS+OS", "PBS"]
    depth_range = [-300, 0]

# ensure the ground truth layers are sorted
ground_truth = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))


## January 2026 - 1070

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

keyword_combinations = [
    ["sim", "average_trace_2026", "no_ap", "ML_decoded", "1fixed_res"],  # Match files with ALL these keywords
    ["sim", "average_trace_2026" ,"600um_no_ap", "1fixed_res"]            # OR files with ALL these keywords
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1070],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1'],
	colors=["black", "cyan", "grey"]
)

## January 2026 - 1070 - no saline fixation

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

keyword_combinations = [
    ["sim", "average_trace_2026", "no_ap", "ML_decoded", "0fixed"],  # Match files with ALL these keywords
    ["sim", "average_trace_2026" ,"600um_no_ap", "0fixed"]            # OR files with ALL these keywords
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1070],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1'],
	colors=["black", "cyan", "grey"]
)

## August 2025 - 1046

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2025", "no_ap", "ML_decoded"],  # Match files with ALL these keywords
    ["sim", "average_trace_2025", "600um_no_ap"]            # OR files with ALL these keywords
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1046],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=["ML-Forward Model-4", "Experimental", "ML-Forward Model-1"],
	colors=["black", "cyan", "grey"]
)

## February 2026 - 1079

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "no_ap", "ML_decoded"],  # Match files with ALL these keywords
    ["sim", "average_trace_2026", "600um_no_ap"]            # OR files with ALL these keywords
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1079],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1'],
	colors=["black", "cyan", "grey"]
)

## February 2026 - 1079 - no saline fixation


In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "no_ap", "ML_decoded", "0fixed_res"],  # Match files with ALL these keywords
    ["sim", "average_trace_2026", "600um_no_ap", "0fixed_res"]            # OR files with ALL these keywords
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1079],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1'],
	colors=["black", "cyan", "grey"]
)

## April 2026 - 1088 - no saline fixation

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "600um_ML_decoded_no_ap", "4fixed", "2depth", "damp"],
	["sim", "average_trace_2026", "600um", "ML_decoded", "no_ap", "4fixed", "1layer", "2depth", "damp"],
    ["sim", "average_trace_2026", "600um_no_ap", "4fixed", "2depth", "damp"]
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# create ground truth based on ML decoded values
ground_truth_list = []
ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))
ground_truth_4l = OrderedDict(sorted({"0": "5922", "-19": "1077", "-67": "3666", "-146": "60", "-200": "60"}.items(),
									 key=lambda x: int(x[0]), reverse=True))
ground_truth_1l = OrderedDict(sorted({"0": "2174", "-19": "2174", "-67": "2174", "-146": "2174", "-200": "60"}.items
									 (), key=lambda x: int(x[0]), reverse=True))
ground_truth_list.append(ground_truth_4l)
ground_truth_list.append(ground_truth_experimental)
ground_truth_list.append(ground_truth_1l)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1088],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1', ],
	colors=["black", "cyan", "grey"]
)

## April 2026 - 1094 - no saline fixation

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "600um_ML_decoded_no_ap", "4fixed", "2depth", "damp"],
	["sim", "average_trace_2026", "600um", "ML_decoded", "4fixed", "1layer_no_ap", "2depth", "damp"],
    ["sim", "average_trace_2026", "600um_no_ap", "4fixed", "2depth", "damp"]
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# create ground truth based on ML decoded values
ground_truth_list = []
ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))
ground_truth_4l = OrderedDict(sorted({"0": "3632", "-19": "1578", "-67": "1249", "-146": "2119", "-200": "60"}.items(),
									 key=lambda x: int(x[0]), reverse=True))
ground_truth_1l = OrderedDict(sorted({"0": "1810", "-19": "1810", "-67": "1810", "-146": "1810", "-200": "60"}.items
									 (), key=lambda x: int(x[0]), reverse=True))
ground_truth_list.append(ground_truth_4l)
ground_truth_list.append(ground_truth_experimental)
ground_truth_list.append(ground_truth_1l)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1094],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels=['ML-Forward Model-4', 'Experimental', 'ML-Forward Model-1'],
	colors=["black", "cyan", "grey", ]
)

## April 2026 - 1097 - no saline fixation

In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "600um_ML_decoded_no_ap", "4fixed", "2depth", "damp"],
	["sim", "average_trace_2026", "600um", "no_ap", "ML_decoded", "4fixed", "1layer", "2depth", "damp"],
    ["sim", "average_trace_2026", "600um_no_ap", "4fixed", "2depth", "damp"]
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# create ground truth based on ML decoded values
ground_truth_list = []
ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))
ground_truth_4l = OrderedDict(sorted({"0": "4874", "-19": "1052", "-67": "2729", "-146": "60", "-200": "60"}.items(),
									 key=lambda x: int(x[0]), reverse=True))
ground_truth_1l = OrderedDict(sorted({"0": "1874", "-19": "1874", "-67": "1874", "-146": "1874", "-200": "60"}.items
									 (), key=lambda x: int(x[0]), reverse=True))
ground_truth_list.append(ground_truth_experimental)
ground_truth_list.append(ground_truth_4l)
ground_truth_list.append(ground_truth_1l)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1097],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels= ['Experimental', 'ML-Forward Model-4', 'ML-Forward Model-1'],
	colors=["cyan", "black", "grey"]
)

## April 2026 - 1097 - no saline fixation


In [ ]:
# initialize the header files list
header_files = CommonUtils.find_header_files(
    root_folder,
    keyword='inversion_result',
    search_depth=1,
    sort_by_index=True
)

# filter by keyword combinations
keyword_combinations = [
    ["sim", "average_trace_2026", "600um", "no_ap", "ML_decoded", "4fixed", "3depth"],
	["sim", "average_trace_2026", "600um", "no_ap", "ML_decoded", "0fixed", "1layer", "3depth"],
    ["sim", "average_trace_2026", "600um_no_ap", "0fixed", "3depth"]
]

# filter the retrieved header files based on the keyword combinations
filtered_files = CommonUtils.filter_by_keyword_combinations(header_files, keyword_combinations)

# create ground truth based on ML decoded values
ground_truth_list = []
ground_truth_experimental = OrderedDict(sorted(layers.items(), key=lambda x: int(x[0]), reverse=True))
ground_truth_4l = OrderedDict(sorted({"0": "4874", "-19": "1052", "-67": "2729", "-146": "60", "-200": "60"}.items(),
									 key=lambda x: int(x[0]), reverse=True))
ground_truth_1l = OrderedDict(sorted({"0": "1874", "-19": "1874", "-67": "1874", "-146": "1874", "-200": "60"}.items
									 (), key=lambda x: int(x[0]), reverse=True))
# ground_truth_list.append(ground_truth_experimental)
# ground_truth_list.append(ground_truth_4l)
ground_truth_list.append(ground_truth_1l)

# convert data to 1D traces
Res2DInvOutputProcessor.run_1d_conversion(
    filtered_files,
    indices_to_process=[1100],
    indices_to_exclude=None,
    x_range=x_range_to_use,
    depth_range=depth_range,
    labels=labels,
    layers=layers,
    ground_truth=ground_truth_list,
    calculate_average_resistivity=True,
    plot_all_together=True,
    x_range_to_plot=[0, 6000],
    legend_labels= ['Experimental', 'ML-Forward Model-4', 'ML-Forward Model-1'],
	colors= ["cyan", "black", "grey"]
)